# 06 — Robustness Analyses

Two independent tests. A supervised classifier assesses whether the factor
combinations are separable in expression space, with cell-type classification as
a positive control. A compositional analysis then asks whether reprogramming
shifts cell-type proportions toward the youthful profile, an axis the network
model cannot represent.

In [ ]:
import sys
!{sys.executable} -m pip install "numpy==1.26.4" scanpy anndata scikit-learn --quiet

from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 81.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 16.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.11.1 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.11.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible

In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd

adata = ad.read_h5ad("/content/drive/MyDrive/roux_project/msc_annotated.h5ad")
adata.X = adata.layers['raw_count'].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

print(adata.obs['combination_short'].value_counts())
print("\ncell types:")
print(adata.obs['cell_type'].value_counts())

combination_short
NT      5452
O        621
M        558
K        519
S        479
OM       282
SM       272
OK       237
SO       233
SK       222
SOM      212
KM       209
SOKM     198
SKM      171
SOK      156
OKM      156
Name: count, dtype: int64

cell types:
cell_type
Fibroblast/stromal        5132
Tendon                    1719
Myofibroblast             1213
Senescent/stressed         754
Proliferating              748
Interferon                 162
Oxidative-stress           134
Inflammatory/secretory      67
Adipogenic                  48
Name: count, dtype: int64


## Combination classifier

A random forest is trained to predict a cell's factor combination from its
expression profile. Convergent combinations should be inseparable, giving
near-chance accuracy. A cell-type classifier trained through the identical
pipeline serves as a positive control: success there alongside failure on the
combination task isolates the failure to the combinations themselves.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# feature matrix: normalised, HVG-subset, scaled
adata_feat = adata.copy()
sc.pp.highly_variable_genes(adata_feat, n_top_genes=2000)
adata_feat = adata_feat[:, adata_feat.var['highly_variable']].copy()
sc.pp.scale(adata_feat, max_value=10)
X = adata_feat.X
print("feature matrix:", X.shape)

# positive control: classify cell type
y_ct = adata_feat.obs['cell_type'].values
Xtr, Xte, ytr, yte = train_test_split(X, y_ct, test_size=0.25, random_state=0, stratify=y_ct)
clf_ct = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
clf_ct.fit(Xtr, ytr)
acc_ct = accuracy_score(yte, clf_ct.predict(Xte))
n_ct = len(np.unique(y_ct))
print(f"\nPOSITIVE CONTROL — cell-type classification")
print(f"  test accuracy: {acc_ct:.3f}   (chance = {1/n_ct:.3f}, {n_ct} classes)")

/usr/lib/python3.13/functools.py:934: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


feature matrix: (9977, 2000)

POSITIVE CONTROL — cell-type classification
  test accuracy: 0.857   (chance = 0.111, 9 classes)


In [ ]:
# treated combinations only, balanced to equal class sizes
treated = adata_feat[adata_feat.obs['combination_short'] != 'NT'].copy()
min_n = treated.obs['combination_short'].value_counts().min()
print(f"balancing to {min_n} cells per combination")

idx_balanced = []
for combo in treated.obs['combination_short'].unique():
    combo_idx = np.where(treated.obs['combination_short'].values == combo)[0]
    idx_balanced.extend(np.random.RandomState(0).choice(combo_idx, min_n, replace=False))
idx_balanced = np.array(idx_balanced)

Xc = treated.X[idx_balanced]
yc = treated.obs['combination_short'].values[idx_balanced]
n_combo = len(np.unique(yc))

Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.25, random_state=0, stratify=yc)
clf_c = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)
clf_c.fit(Xtr, ytr)
pred = clf_c.predict(Xte)

print(f"\nCOMBINATION CLASSIFICATION ({n_combo} treated combinations, balanced)")
print(f"  test accuracy:     {accuracy_score(yte, pred):.3f}")
print(f"  balanced accuracy: {balanced_accuracy_score(yte, pred):.3f}")
print(f"  chance level:      {1/n_combo:.3f}")
print(f"\nContrast: cell type {acc_ct:.3f} vs combination {accuracy_score(yte, pred):.3f}")

balancing to 156 cells per combination

COMBINATION CLASSIFICATION (15 treated combinations, balanced)
  test accuracy:     0.111
  balanced accuracy: 0.111
  chance level:      0.067

Contrast: cell type 0.857 vs combination 0.111


## Compositional analysis

Ageing in this dataset is accompanied by a shift in cell-type proportions toward
activated and stressed populations. Because the network model represents only
per-cell expression and not composition, this provides an independent test of
rejuvenation: if reprogramming restores a youthful state, treated aged cells
should adopt a composition closer to the young reference.

In [ ]:
adata.obs['treated'] = np.where(adata.obs['combination_short']=='NT', 'NT', 'Treated')
print("cell counts by age and treatment status:")
print(pd.crosstab(adata.obs['age'], adata.obs['treated']))

comp = pd.crosstab(adata.obs['cell_type'],
                   [adata.obs['age'], adata.obs['treated']],
                   normalize='columns')
comp.columns = [f"{a}_{t}" for a, t in comp.columns]
print("\ncell-type composition (proportions):\n")
print(comp.round(3).to_string())

cell counts by age and treatment status:
treated    NT  Treated
age                   
Young    4460     3084
Aged      992     1441

cell-type composition (proportions):

                        Young_NT  Young_Treated  Aged_NT  Aged_Treated
cell_type                                                             
Adipogenic                 0.008          0.004    0.001         0.000
Fibroblast/stromal         0.620          0.446    0.438         0.387
Inflammatory/secretory     0.000          0.000    0.046         0.015
Interferon                 0.022          0.015    0.013         0.005
Myofibroblast              0.092          0.080    0.238         0.221
Oxidative-stress           0.002          0.020    0.005         0.040
Proliferating              0.082          0.046    0.124         0.081
Senescent/stressed         0.019          0.126    0.054         0.157
Tendon                     0.156          0.263    0.081         0.094


In [ ]:
young_baseline = comp['Young_NT']

print("Euclidean distance of each group's composition from Young_NT:\n")
for grp in ['Aged_NT', 'Aged_Treated', 'Young_Treated']:
    dist = np.sqrt(((comp[grp] - young_baseline)**2).sum())
    print(f"  {grp:15} {dist:.4f}")

print("\nIf reprogramming restored a youthful composition, Aged_Treated would sit")
print("closer to Young_NT than Aged_NT does.")

Euclidean distance of each group's composition from Young_NT:

  Aged_NT         0.2562
  Aged_Treated    0.3097
  Young_Treated   0.2352

If reprogramming restored a youthful composition, Aged_Treated would sit
closer to Young_NT than Aged_NT does.


## Summary

The combination classifier reached 11.1% accuracy (chance 6.7%) while the
identical pipeline classified cell type at 85.7% (chance 11.1%), indicating that
the factor combinations do not carry separable transcriptional signatures.

Compositionally, aged treated cells were further from the young reference than
aged untreated cells, driven by an increased proportion of senescent/stressed
and oxidative-stress populations and a reduced proportion of resting
fibroblasts. The same shift was present in young treated cells, indicating it
may reflect a response to the perturbation itself rather than reprogramming
specifically.